# 03 - GNN Training (GraphSAGE vs GCN)
Trains GraphSAGE and compares with GCN baseline; logs comparison to MLflow.

In [ ]:
from pathlib import Path
import os
import sys
import mlflow
import numpy as np
import pandas as pd
import torch
from torch_geometric.nn import GCNConv

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from falabella_risk.models.train_gnn import (
    GraphSAGEClassifier,
    build_graph_tensors,
    compute_metrics,
    split_indices,
    train_graphsage,
)

train_graphsage(
    feature_path=Path('data/processed/features.parquet'),
    edges_path=Path('data/raw/edges.parquet'),
    labels_path=Path('data/raw/labels.parquet'),
    model_out=Path('models/graphsage.pt'),
    embeddings_out=Path('data/processed/gnn_embeddings.parquet'),
    seed=42,
    epochs=35,
    learning_rate=0.003,
)

In [ ]:
features = pd.read_parquet('data/processed/features.parquet')
edges = pd.read_parquet('data/raw/edges.parquet')
labels = pd.read_parquet('data/raw/labels.parquet')
x, edge_index, y, _ = build_graph_tensors(features, edges, labels)
train_idx, val_idx, test_idx = split_indices(len(x), seed=42)

class GCNClassifier(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=128):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.out = torch.nn.Linear(hidden_channels, 2)

    def forward(self, x_t, ei):
        h = torch.relu(self.conv1(x_t, ei))
        h = torch.relu(self.conv2(h, ei))
        return self.out(h)

gcn = GCNClassifier(in_channels=x.shape[1], hidden_channels=128)
opt = torch.optim.Adam(gcn.parameters(), lr=0.003, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss()
best_auc = -1.0
best_state = None
for epoch in range(35):
    gcn.train()
    opt.zero_grad()
    logits = gcn(x, edge_index)
    loss = criterion(logits[train_idx], y[train_idx])
    loss.backward()
    opt.step()

    with torch.no_grad():
        val_logits = gcn(x, edge_index)
        val_metrics = compute_metrics(val_logits, y, val_idx)
    if val_metrics['auc'] > best_auc:
        best_auc = val_metrics['auc']
        best_state = {k: v.detach().clone() for k, v in gcn.state_dict().items()}

gcn.load_state_dict(best_state)
with torch.no_grad():
    gcn_logits = gcn(x, edge_index)
    gcn_test = compute_metrics(gcn_logits, y, test_idx)

gs = GraphSAGEClassifier(in_channels=x.shape[1], hidden_channels=128)
gs.load_state_dict(torch.load('models/graphsage.pt', map_location='cpu'))
gs.eval()
with torch.no_grad():
    gs_logits, _ = gs(x, edge_index)
    gs_test = compute_metrics(gs_logits, y, test_idx)

mlflow.set_experiment('falabella_gnn_training')
with mlflow.start_run(run_name='graphsage_vs_gcn_compare'):
    mlflow.log_metric('graphsage_auc', float(gs_test['auc']))
    mlflow.log_metric('graphsage_pr_auc', float(gs_test['pr_auc']))
    mlflow.log_metric('gcn_auc', float(gcn_test['auc']))
    mlflow.log_metric('gcn_pr_auc', float(gcn_test['pr_auc']))

pd.DataFrame([{'model': 'GraphSAGE', **gs_test}, {'model': 'GCN', **gcn_test}])